# Why do stock markets make their money *overnight*? 🌙
### A real market anomaly, what's at stake, and why it's subtler than it looks

Over the last 30 years, almost all the gains of the world's big stock markets piled up **while the market was closed** — from one day's close to the next morning's open. The **daytime session** (open → close) is nearly flat. Someone even claims this is the fingerprint of a giant fraud.

This notebook tells the story **without jargon**, explains **why it matters**, and shows why the truth is more interesting than the headline. For the rigorous version (statistics, microstructure, capacity), see [`02_for_the_quants.ipynb`](02_for_the_quants.ipynb).

> ⚠️ **This is not investment advice.** Educational and research tool.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")


## Why should you care? (the stakes)

This isn't a trivia question. Three very different things hang on the answer:

- **A fraud accusation.** One researcher (Bruce Knuteson) argues the pattern is the signature of a single huge hedge fund secretly manipulating the world's markets — a *multi-trillion-dollar* claim against the financial system. If true, it's one of the biggest scandals in finance. If not, it's a careless accusation. **Which it is depends entirely on the numbers.**
- **Your savings.** If "stocks only go up at night" were a free lunch, you could just buy at the close and sell at the open. Funds tried exactly that (the *NSPY* and *NIWM* ETFs). They launched in 2022 and were **shut down in 2023** after losing to the market. Understanding *why* protects you from the next shiny promise.
- **How markets actually work.** The pattern is real and well documented. The interesting question is *what causes it* — and the honest answer teaches you more about markets than any get-rich scheme.

Our job here: separate **what's true** from **what's exaggerated** from **what's unproven** — with code you can run yourself.

## Why would a market move at night *at all*?

Before crying fraud, notice there are perfectly ordinary reasons the *closed* hours could carry most of the return:

1. **News doesn't wait for the bell.** Companies report earnings *after* the close; economic data and overseas events land overnight. Prices jump at the next open to absorb it.
2. **The world keeps trading while you sleep.** When New York is closed, Tokyo and London are open. A US-listed fund on foreign stocks will do most of its *real* moving during America's night.
3. **You get paid to hold risk you can't escape.** Holding stocks overnight means bearing gap risk while you *cannot* trade. Markets tend to pay a premium for bearing risk — so some overnight return is just **compensation for risk**, not magic.
4. **The night is simply longer.** The closed window (evening + the whole next morning, plus entire weekends) spans far more calendar time than the 6.5-hour trading day. More hours → more drift. *(The quant notebook makes this precise — and it dissolves much of the mystery.)*

Keep these in mind: each is a boring, fraud-free reason the night can win.

## 1. The pattern: the night rises, the day stalls

Let's build a *toy* market that is completely honest: each night it drifts a hair upward (+3 basis points, i.e. +0.03%), each day a hair downward, and **everything else is pure noise — no fraud, no conspiracy**. What does the night/day decomposition show?

In [ ]:
from overnight import decompose, diagnostics

ohlc = diagnostics.synthetic_ohlc(overnight_bias_bps=3, intraday_bias_bps=-1, seed=0)
dec = decompose.decompose(ohlc)

ax = plt.subplot()
ax.plot(dec.index, dec['cum_overnight']*100, label='Overnight (close→open)', lw=2)
ax.plot(dec.index, dec['cum_intraday']*100, label='Intraday (open→close)', lw=2)
ax.plot(dec.index, dec['cum_close_close']*100, label='Buy & hold', color='grey', lw=1.2)
ax.set_title('Toy market — no fraud, just a 3 bps overnight bias')
ax.set_ylabel('Cumulative return (%)'); ax.legend(); ax.grid(alpha=.3)
plt.show()
s = decompose.summary(dec)
print(f"Overnight cumulative: {s.loc['overnight','cum_return']*100:+.0f}%   "
      f"Intraday cumulative: {s.loc['intraday','cum_return']*100:+.0f}%")

**We just reproduced the scary pattern with zero manipulation.** A tiny constant bias, repeated ~250 nights a year for decades, is enough. The night *looks* magical. Now three traps that inflate the story — then the punchline.

## 2. Trap #1 — the scale lies (the magic of compounding)

The headline charts use a **logarithmic** scale, where you quickly read "billions of %". Where does that dizzying number come from? Not fraud: **compounding**. Look at what a simple constant bias becomes, by size and horizon:

In [ ]:
table = diagnostics.compounding_table()
diagnostics.format_compounding(table)

A bias of **1 basis point per night** — totally innocent, undetectable — compounds to three digits over 30 years. At 30 bps, you reach *trillions* of percent. **The explosion comes from the exponent, not from a conspiracy.**

## 3. Trap #2 — dirty data manufactures the signal

Free data (Yahoo) mis-handles some *splits* and dividends, especially in emerging markets. A handful of corrupted prices is enough to **mechanically shift return from the day into the night**. On a perfectly flat market we dirty just 3 prices:

In [ ]:
flat = diagnostics.synthetic_ohlc(overnight_bias_bps=0, intraday_bias_bps=0, seed=1)
clean = decompose.decompose(flat)
dirty = decompose.decompose(diagnostics.inject_split_artifact(flat, factor=1.5))
print(f"Overnight cumulative  BEFORE: {clean['cum_overnight'].iloc[-1]*100:+.1f}%")
print(f"Overnight cumulative  AFTER : {dirty['cum_overnight'].iloc[-1]*100:+.1f}%   (3 dirtied prices)")
flags = diagnostics.flag_suspicious_returns(dirty)
print(f"\nThe automatic detector flags {len(flags)} suspicious day(s).")

Three data errors, and the "overnight performance" flips from red to bright green. This is the mechanism behind the wildest emerging-market numbers. **Before crying scandal, check your data.**

## 4. Trap #3 — fees erase the gain

Suppose the night effect is real (it partly is). Can you *trade* it? Buying every close and selling every open pays the spread **twice a day, ~250 days a year**. What's left after realistic costs?

In [ ]:
from overnight import backtest
sweep = backtest.cost_sweep(dec, roundtrip_bps=(0,1,2,3,5,8))
view = sweep.copy()
view['cagr_net'] = (view['cagr_net']*100).map('{:+.2f}%'.format)
view['sharpe_net'] = view['sharpe_net'].map('{:+.2f}'.format)
view['max_drawdown'] = (view['max_drawdown']*100).map('{:.0f}%'.format)
view.columns = ['Net annual return', 'Net Sharpe', 'Worst drawdown']
view.index.name = 'Round-trip cost (bps)'
view

At **0 fees**, the Sharpe is decent (~0.7). At a realistic **5 basis points** round-trip, the gain turns **negative**. This is exactly what sank the NSPY / NIWM "night effect" ETFs: launched June 2022, **liquidated August 2023**.

> *A strategy that's beautiful on paper is worth no more than the paper until it has paid the real costs of execution.*

## So… fraud, or not?

Here's the honest verdict, and why it's more interesting than the headline:

| | |
|---|---|
| ✅ **The fact is real** | the night really did outperform the day, for decades |
| ⚠️ **But the numbers are inflated** | compounding + log scale + dirty data |
| 🕐 **And partly an illusion** | the night is simply *longer* — see the quant notebook |
| ❌ **And not tradable** | fees erase the edge (RIP, night ETFs) |
| 🔍 **Fraud? Not proven** | ordinary risk + market plumbing explain it just as well |

The pattern is genuine and fascinating — but "the market is rigged at night" is a much bigger claim than the evidence supports. The quant notebook shows, with proper statistics, *why a careful researcher stays sceptical*: [`02_for_the_quants.ipynb`](02_for_the_quants.ipynb).

*Further reading:* the original articles and the academic literature (with a map of who argues what) are in [`docs/references.md`](../docs/references.md); grab the PDFs with `python papers/download_papers.py`.